In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from ultralytics import YOLO

# ==========================================
# 1. Configurar Diretórios
# ==========================================
BASE_DIR = "../data/test"
IMAGE_DIR = os.path.join(BASE_DIR, "image")
CLOTH_DIR = os.path.join(BASE_DIR, "cloth")
MASK_DIR = os.path.join(BASE_DIR, "cloth-mask")

# Obter todos os ficheiros da pasta de imagens (ordenados)
lista_modelos = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))])

print(f"Iniciando Batch Test: Encontrados {len(lista_modelos)} pares de imagens para processar.\n")
pose_model = YOLO("yolov8n-pose.pt")

# ==========================================
# 2. Loop de Processamento (O Verdadeiro Teste)
# ==========================================
for ficheiro in lista_modelos:
    print(f"A processar: {ficheiro}...")
    
    img_path = os.path.join(IMAGE_DIR, ficheiro)
    cloth_path = os.path.join(CLOTH_DIR, ficheiro)
    
    # As máscaras no VITON-HD costumam ser .png mesmo que as fotos sejam .jpg
    nome_mascara = ficheiro.replace(".jpg", ".png")
    mask_path = os.path.join(MASK_DIR, nome_mascara)
    
    # Verificação de segurança: garantir que os 3 ficheiros existem
    if not os.path.exists(cloth_path) or not os.path.exists(mask_path):
        print(f"Faltam ficheiros (Roupa ou Máscara) para o modelo {ficheiro}. A saltar...\n")
        continue

    # Ler Imagens
    img_modelo = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img_roupa_original = cv2.cvtColor(cv2.imread(cloth_path), cv2.COLOR_BGR2RGB)
    img_mascara = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Tratamento da Roupa (Remover Fundo)
    roupa_transparente = cv2.cvtColor(img_roupa_original, cv2.COLOR_RGB2RGBA)
    _, mascara_binaria = cv2.threshold(img_mascara, 127, 255, cv2.THRESH_BINARY)
    roupa_transparente[:, :, 3] = mascara_binaria 

    # Extração Anatómica (YOLO)
    resultados = pose_model(img_modelo, verbose=False)
    
    # Se não vir a pessoa, passa à próxima
    if len(resultados[0].keypoints) == 0:
        print(f"Falha: Nenhuma pessoa detetada em {ficheiro}\n")
        continue
        
    keypoints = resultados[0].keypoints.xy.cpu().numpy()[0]
    nariz = keypoints[0]           
    ombro_esq = keypoints[5] if keypoints[5][0] < keypoints[6][0] else keypoints[6]
    ombro_dir = keypoints[6] if keypoints[5][0] < keypoints[6][0] else keypoints[5]

    if ombro_esq[0] == 0 or ombro_dir[0] == 0:
        print(f"Falha: Ombros não visíveis em {ficheiro}\n")
        continue

    # Matemática do Encaixe
    img_resultado = img_modelo.copy()
    dist_ombros = abs(ombro_dir[0] - ombro_esq[0])
    centro_y_ombros = (ombro_esq[1] + ombro_dir[1]) / 2

    largura_alvo = int(dist_ombros * 2.1)
    proporcao = roupa_transparente.shape[0] / roupa_transparente.shape[1]
    altura_alvo = int(largura_alvo * proporcao)

    roupa_redimensionada = cv2.resize(roupa_transparente, (largura_alvo, altura_alvo), interpolation=cv2.INTER_AREA)

    centro_x = int((ombro_esq[0] + ombro_dir[0]) / 2)
    pos_x = centro_x - (largura_alvo // 2)

    if nariz[1] != 0:
        y_pescoco = nariz[1] + ((centro_y_ombros - nariz[1]) * 0.6)
        pos_y = int(y_pescoco) - int(altura_alvo * 0.05)
    else:
        pos_y = int(centro_y_ombros) - int(altura_alvo * 0.1)

    # Colagem (Alpha Blending)
    h, w, _ = img_resultado.shape
    y1, y2 = max(0, pos_y), min(h, pos_y + altura_alvo)
    x1, x2 = max(0, pos_x), min(w, pos_x + largura_alvo)

    roupa_y1 = max(0, -pos_y)
    roupa_y2 = roupa_y1 + (y2 - y1)
    roupa_x1 = max(0, -pos_x)
    roupa_x2 = roupa_x1 + (x2 - x1)

    try:
        alpha_mask = roupa_redimensionada[roupa_y1:roupa_y2, roupa_x1:roupa_x2, 3] / 255.0
        img_modelo_float = img_resultado[y1:y2, x1:x2].astype(float)
                    
        for c in range(0, 3):
            img_modelo_float[:, :, c] = (
                alpha_mask * roupa_redimensionada[roupa_y1:roupa_y2, roupa_x1:roupa_x2, c] +
                (1 - alpha_mask) * img_modelo_float[:, :, c]
            )
                    
        img_resultado[y1:y2, x1:x2] = img_modelo_float.astype(np.uint8)
    except Exception as e:
        print(f"Erro geométrico em {ficheiro}: {e}\n")
        continue

    # Plot Visual de cada iteração
    fig, axs = plt.subplots(1, 3, figsize=(14, 5))

    axs[0].imshow(img_modelo)
    axs[0].set_title(f"Modelo: {ficheiro}")
    axs[0].axis('off')

    axs[1].imshow(roupa_transparente)
    axs[1].set_title(f"Roupa (Fundo Removido)")
    axs[1].axis('off')

    axs[2].imshow(img_resultado)
    axs[2].set_title(f"Resultado Automático")
    axs[2].axis('off')

    plt.tight_layout()
    plt.show()
    
print("Test Concluído com Sucesso!")

Iniciando Batch Test: Encontrados 15 pares de imagens para processar.

A processar: 00035_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00035_00.jpg. A saltar...

A processar: 00055_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00055_00.jpg. A saltar...

A processar: 00057_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00057_00.jpg. A saltar...

A processar: 00064_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00064_00.jpg. A saltar...

A processar: 00067_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00067_00.jpg. A saltar...

A processar: 00069_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00069_00.jpg. A saltar...

A processar: 00071_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00071_00.jpg. A saltar...

A processar: 00074_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00074_00.jpg. A saltar...

A processar: 00075_00.jpg...
Faltam ficheiros (Roupa ou Máscara) para o modelo 00